In [2]:
import pandas as pd
import joblib
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split, cross_val_score
from catboost import CatBoostClassifier, Pool

# load & prepare data
data = pd.read_csv('../../CSV/customer_churn_lightgbm.csv')
X = data.drop(columns='churn')
y = data['churn']

print("[ DATA ]")
print(data)

# split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

model = CatBoostClassifier(
    iterations=500,       # number of boosting rounds
    depth=6,              # tree depth
    learning_rate=0.1,
    loss_function='Logloss',
    verbose=0,            # silent output (set to 100 if you want logs every 100 iterations)
    random_seed=42
)
model.fit(X_train, y_train)

# test model by making predictions
# Get predicted probabilities
probs = model.predict_proba(X_test)[:, 1]

# Adjust threshold to reduce false negatives or false positives
threshold = 0.5 # tweak this to balance FP and FN
predictions = (probs > threshold).astype(int)

# evaluate model
accuracy = accuracy_score(y_test, predictions)

# save model
joblib.dump(model, 'models/customer_churn_predictor[LightGBM].joblib')

# 1. Accuracy - Bar Plot
print("[ ACCURACY ]")
plt.figure(figsize=(4, 4))
plt.bar(["Accuracy"], [accuracy], color='skyblue')
plt.ylim(0, 1)
plt.title("Model Accuracy")
plt.ylabel("Score")
plt.show()


# 2. Actual vs Predicted - Bar Chart
results_df = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': predictions
})

plt.figure(figsize=(8, 4))
sns.countplot(
    x='value', hue='Type',
    data=pd.melt(results_df, value_vars=['Actual', 'Predicted'], var_name='Type'),
    palette='pastel'
)
plt.title("Actual vs Predicted Class Counts")
plt.xlabel("Class")
plt.ylabel("Count")
plt.legend(title='Label Type')
plt.show()

# 3. Confusion Matrix - Heatmap
cm = confusion_matrix(y_test, predictions)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Pred 0', 'Pred 1'], yticklabels=['Actual 0', 'Actual 1'])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# 4. Precision, Recall, F1 Score - Heatmap
report = classification_report(y_test, predictions, output_dict=True, zero_division=0)

# Convert to DataFrame (remove 'accuracy' to keep it 2D for heatmap)
report_df = pd.DataFrame(report).drop(columns=['accuracy'])
plt.figure(figsize=(8, 5))
sns.heatmap(report_df.iloc[:3, :].T, annot=True, cmap="YlGnBu", fmt=".2f")
plt.title("Classification Report")
plt.ylabel("Class")
plt.xlabel("Metrics")
plt.tight_layout()
plt.show()

# 5. Cross-Validation Scores - Bar Plot
scores = cross_val_score(model, X, y, cv=3)
plt.figure(figsize=(8, 4))
plt.bar(range(1, 4), scores, color='lightgreen')
plt.ylim(0, 1)
plt.title("Cross-Validation Accuracy per Fold")
plt.xlabel("Fold")
plt.ylabel("Accuracy")
plt.xticks(range(1, 4))
plt.show()

lgb.plot_importance(model, importance_type='gain', max_num_features=10)
plt.title("Top 10 Feature Importances (by Gain)")
plt.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'CSV/customer_churn_lightgbm.csv'